# 1. Creating table

In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.dim_product AS

WITH distinct_products AS (
    SELECT DISTINCT
        series_identifier,
        COALESCE(category, 'Unknown')    AS category,
        COALESCE(subcategory, 'Unassigned') AS subcategory,
        COALESCE(survey_code, 'Unknown') AS survey_code
    FROM cpt_utility_catalog.silver.silver_cpi_cleaned
    WHERE series_identifier IS NOT NULL
),

products_with_keys AS (
    SELECT
        -- Standardized composite hash key
        xxhash64(concat_ws('||', 
            LOWER(TRIM(series_identifier)), 
            LOWER(TRIM(category)), 
            LOWER(TRIM(subcategory)), 
            LOWER(TRIM(survey_code))
        )) AS product_key,
        series_identifier,
        category,
        subcategory,
        survey_code,
        TRUE AS is_active
    FROM distinct_products
)

SELECT * FROM products_with_keys

UNION ALL

-- Mandatory Kimball Fallback Record
SELECT
    xxhash64('unmapped') AS product_key,
    'Unmapped'           AS series_identifier,
    'Unmapped'           AS category,
    'Unmapped'           AS subcategory,
    'Unmapped'           AS survey_code,
    TRUE                 AS is_active;

In [0]:
%sql
SELECT * FROM cpt_utility_catalog.gold.dim_product